# LSTM Autoencoder — NIDS Training

Train multiple LSTM-AE configurations on Monday benign traffic from CICIDS2017.

**Key idea:** LSTM processes the 78 features as a sequence, capturing
sequential dependencies between flow features. The encoder compresses
the sequence into a fixed-length vector, and the decoder reconstructs it.

In [1]:
import sys, os

# Mount Google Drive and add the project directory to Python path
from google.colab import drive
drive.mount("/content/drive")

NIDS_ROOT = "/content/drive/MyDrive/Colab Notebooks/nids"
sys.path.insert(0, NIDS_ROOT)

import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt

import tensorflow as tf
import keras
from tensorflow.keras import layers, callbacks

import joblib
import nids_utils as nu

print(f"TF version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

Mounted at /content/drive
TF version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 1. Load & Preprocess Data

In [2]:
SCALER_SAVE = os.path.join(nu.PREPROCESSING_DIR, "scaler_lstm_ae.pkl")

X_train, scaler, feature_cols = nu.prepare_train_data(
    monday_path=nu.MONDAY_CSV,
    scaler_save_path=SCALER_SAVE,
)

input_dim = X_train.shape[1]
print(f"Training samples: {X_train.shape[0]}, features: {input_dim}")

# Reshape to (samples, timesteps, 1) for LSTM
X_train_lstm = X_train.reshape(-1, input_dim, 1)
print(f"LSTM input shape: {X_train_lstm.shape}")

Loaded /content/drive/MyDrive/Colab Notebooks/nids/CICIDS2017-dataset/Monday-WorkingHours.pcap_ISCX.csv  shape=(529918, 79)
Label distribution:
Label
BENIGN    529918
Name: count, dtype: int64
Scaler saved → /content/drive/MyDrive/Colab Notebooks/nids/preprocessing_tools/scaler_lstm_ae.pkl
Training samples: 529918, features: 78
LSTM input shape: (529918, 78, 1)


## 2. LSTM-AE Model Definition

In [3]:
def build_lstm_ae(
    timesteps, n_features, enc_units, dec_units, dropout, bidirectional
):
    """
    Build an LSTM autoencoder.

    Parameters
    ----------
    timesteps      : int   – sequence length (78 features)
    n_features     : int   – channels per timestep (1)
    enc_units      : list[int] – LSTM units for each encoder layer
    dec_units      : list[int] – LSTM units for each decoder layer
    dropout        : float
    bidirectional  : bool  – wrap encoder LSTMs in Bidirectional
    """
    inp = layers.Input(shape=(timesteps, n_features))
    x = inp

    # ---------- Encoder ----------
    for i, units in enumerate(enc_units):
        return_seq = i < len(enc_units) - 1  # last layer returns single vector
        lstm = layers.LSTM(units, return_sequences=return_seq, dropout=dropout)
        x = layers.Bidirectional(lstm)(x) if bidirectional else lstm(x)

    # latent vector
    latent = x

    # ---------- Decoder ----------
    x = layers.RepeatVector(timesteps)(latent)

    for units in dec_units:
        lstm = layers.LSTM(units, return_sequences=True, dropout=dropout)
        x = lstm(x)

    out = layers.TimeDistributed(layers.Dense(n_features))(x)

    model = keras.Model(inp, out, name="lstm_autoencoder")
    return model

## 3. Configuration Grid

In [4]:
CONFIGS = {
    "lstm_ae_small_32": {
        "enc_units": [32],
        "dec_units": [32],
        "dropout": 0.1,
        "bidirectional": False,
        "batch_size": 256,
        "lr": 1e-3,
        "epochs": 80,
        "patience": 8,
    },
    "lstm_ae_medium_64_32": {
        "enc_units": [64, 32],
        "dec_units": [32, 64],
        "dropout": 0.15,
        "bidirectional": False,
        "batch_size": 256,
        "lr": 1e-3,
        "epochs": 80,
        "patience": 8,
    },
    "lstm_ae_bidir_64": {
        "enc_units": [64],
        "dec_units": [64],
        "dropout": 0.1,
        "bidirectional": True,
        "batch_size": 256,
        "lr": 1e-3,
        "epochs": 80,
        "patience": 8,
    },
    "lstm_ae_deep_bidir_128_64": {
        "enc_units": [128, 64],
        "dec_units": [64, 128],
        "dropout": 0.2,
        "bidirectional": True,
        "batch_size": 512,
        "lr": 5e-4,
        "epochs": 80,
        "patience": 8,
    },
}

## 4. Training Loop

In [ ]:
MODEL_FAMILY = "lstm_ae"


def reshape_for_lstm(X):
    return X.reshape(-1, X.shape[1], 1) if X.ndim == 2 else X


for cfg_name, cfg in CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"  Training: {cfg_name}")
    print(f"{'='*60}")

    model_dir = nu.get_model_dir(MODEL_FAMILY, cfg_name)

    model = build_lstm_ae(
        timesteps=input_dim,
        n_features=1,
        enc_units=cfg["enc_units"],
        dec_units=cfg["dec_units"],
        dropout=cfg["dropout"],
        bidirectional=cfg["bidirectional"],
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=cfg["lr"]),
        loss="mse",
    )
    model.summary()

    early_stop = callbacks.EarlyStopping(
        monitor="val_loss", patience=cfg["patience"], restore_best_weights=True
    )

    history = model.fit(
        X_train_lstm, X_train_lstm,
        epochs=cfg["epochs"],
        batch_size=cfg["batch_size"],
        validation_split=0.1,
        shuffle=True,
        callbacks=[early_stop],
    )

    # --- Reconstruction error & threshold ---
    recon = model.predict(X_train_lstm, batch_size=1024, verbose=0)
    train_errors = nu.reconstruction_mse(X_train_lstm, recon)
    threshold = nu.compute_threshold(train_errors, method="percentile", percentile=97)

    # --- Save ---
    model.save(os.path.join(model_dir, f"{cfg_name}.keras"))

    nu.save_training_artifacts(
        model_dir, history, threshold,
        extra_meta={
            "config": cfg, "input_dim": input_dim,
            "model_type": "lstm_ae", "input_shape": "(N, 78, 1)",
        },
    )
    nu.plot_error_distribution(
        train_errors, threshold,
        title=f"{cfg_name} — Train Error Distribution",
        save_path=os.path.join(model_dir, "error_dist.png"),
    )

    print(f"  Threshold (97th pctl): {threshold:.6f}")
    print(f"  Mean train MSE:        {train_errors.mean():.6f}")

print("\nAll LSTM-AE configs trained and saved.")


  Training: lstm_ae_small_32


Model: "lstm_autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 78, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 32)             │         4,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 78, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 78, 32)         │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 78, 1)          │            33 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,705 (49.63 KB)

 Trainable params: 12,705 (49.63 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 78, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 78, 256)        │       133,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_3 (RepeatVector)  │ (None, 78, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_10 (LSTM)                  │ (None, 78, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_11 (LSTM)                  │ (None, 78, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, 78, 1)          │           129 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 445,825 (1.70 MB)

 Trainable params: 445,825 (1.70 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/80
932/932 ━━━━━━━━━━━━━━━━━━━━ 106s 105ms/step - loss: 0.0476 - val_loss: 0.0055
Epoch 2/80
932/932 ━━━━━━━━━━━━━━━━━━━━ 96s 103ms/step - loss: 0.0195 - val_loss: 0.0036
Epoch 3/80
932/932 ━━━━━━━━━━━━━━━━━━━━ 95s 102ms/step - loss: 0.0063 - val_loss: 0.0033
Epoch 4/80
932/932 ━━━━━━━━━━━━━━━━━━━━ 95s 102ms/step - loss: 0.0073 - val_loss: 0.0025
Epoch 5/80
932/932 ━━━━━━━━━━━━━━━━━━━━ 96s 103ms/step - loss: 0.0053 - val_loss: 0.0048
Epoch 6/80
932/932 ━━━━━━━━━━━━━━━━━━━━ 95s 102ms/step - loss: 0.0069 - val_loss: 0.0021
Epoch 7/80
932/932 ━━━━━━━━━━━━━━━━━━━━ 95s 102ms/step - loss: 0.0029 - val_loss: 9.3456e-04
Epoch 8/80
932/932 ━━━━━━━━━━━━━━━━━━━━ 95s 102ms/step - loss: 0.0021 - val_loss: 5.2993e-04
Epoch 9/80
932/932 ━━━━━━━━━━━━━━━━━━━━ 142s 102ms/step - loss: 0.0017 - val_loss: 4.9794e-04
Epoch 10/80
932/932 ━━━━━━━━━━━━━━━━━━━━ 95s 102ms/step - loss: 0.0014 - val_loss: 3.8746e-04
Epoch 11/80
932/932 ━━━━━━━━━━━━━━━━━━━━ 95s 102ms/step - loss: 0.0094 - val_loss: 6.9895e-

## 5. Quick Evaluation (All Models)

Evaluate every trained config on the attack files and produce a combined comparison.

In [ ]:
all_eval_dfs = []

for cfg_name in CONFIGS:
    print(f"\n{'='*60}")
    print(f"  Evaluating: {cfg_name}")
    print(f"{'='*60}")

    model_dir = nu.get_model_dir(MODEL_FAMILY, cfg_name)
    model_path = os.path.join(model_dir, f"{cfg_name}.keras")
    threshold_path = os.path.join(model_dir, "threshold.json")

    # Load saved model and threshold
    saved_model = keras.models.load_model(model_path)
    with open(threshold_path) as f:
        saved_threshold = json.load(f)["threshold"]

    # Evaluate on attack files
    eval_df = nu.evaluate_on_attack_files(
        model=saved_model,
        scaler=scaler,
        feature_cols=feature_cols,
        threshold=saved_threshold,
        reshape_fn=reshape_for_lstm,
    )
    eval_df.insert(0, "Model", cfg_name)

    # Save per-model quick_eval
    eval_df.to_csv(os.path.join(model_dir, "quick_eval.csv"), index=False)
    print(eval_df.to_string(index=False))

    all_eval_dfs.append(eval_df)

    del saved_model  # free memory

# ── Combined comparison across all models ──
combined_df = pd.concat(all_eval_dfs, ignore_index=True)

# Summary: average metrics per model
summary = combined_df.groupby("Model").agg({
    "Accuracy": "mean", "Precision": "mean", "Recall": "mean",
    "F1": "mean", "AUC": "mean",
}).reset_index().sort_values("F1", ascending=False)
summary.columns = ["Model", "Avg_Accuracy", "Avg_Precision", "Avg_Recall", "Avg_F1", "Avg_AUC"]

# Save combined results
family_dir = os.path.join(nu.MODELS_ROOT, MODEL_FAMILY)
combined_df.to_csv(os.path.join(family_dir, "all_models_quick_eval.csv"), index=False)
summary.to_csv(os.path.join(family_dir, "model_comparison_summary.csv"), index=False)

print(f"\n{'='*60}")
print("  MODEL COMPARISON SUMMARY (sorted by Avg F1)")
print(f"{'='*60}")
print(summary.to_string(index=False))
print(f"\nCombined results saved → {os.path.join(family_dir, 'all_models_quick_eval.csv')}")
print(f"Summary saved → {os.path.join(family_dir, 'model_comparison_summary.csv')}")